# Agentic AI for Your Lakehouse: Snowflake Cortex, and Gemini Enterprise on Iceberg

We're building an AI-ready data product on open Iceberg — from raw Marketplace data to a Cortex Agent that any AI client can query. By the end, the same agent answers questions in Snowflake CoWork, in Gemini Enterprise, and through any MCP-compatible tool. One copy of data, one agent, many surfaces.

The core idea: business logic belongs in the data layer, not in prompts. Define it once in a Semantic View, ground your agent in it, and every consumer — human or AI — gets the same correct answer.

## Architecture

We're building an American Wellbeing Tracker agent. We pull data from Marketplace, join it into an Iceberg table with clear dimensions and metrics, explore the dataset, create a Semantic View and Cortex Agent powered by Gemini, then expose that agent to business users through CoWork and to the entire organization through Gemini Enterprise via MCP.

In [ ]:
from IPython.display import display, HTML
import base64, pathlib

def spotlight(image_path, left=0, top=0, right=100, bottom=100):
    """Display an image with a dark overlay and a bright cutout region (percentages)."""
    svg_data = pathlib.Path(image_path).read_text()
    b64 = base64.b64encode(svg_data.encode()).decode()
    img_src = f"data:image/svg+xml;base64,{b64}"
    display(HTML(f'''
    <div style="position:relative; width:100%; line-height:0;">
      <img src="{img_src}" style="width:100%; display:block;" />
      <svg style="position:absolute; top:0; left:0; width:100%; height:100%;" viewBox="0 0 100 100" preserveAspectRatio="none">
        <defs>
          <mask id="spotlight">
            <rect x="0" y="0" width="100" height="100" fill="white" />
            <rect x="{left}" y="{top}" width="{right-left}" height="{bottom-top}" fill="black" />
          </mask>
        </defs>
        <rect x="0" y="0" width="100" height="100" fill="black" fill-opacity="0.25" mask="url(#spotlight)" />
      </svg>
    </div>
    '''))

# Full architecture — no spotlight
spotlight("arch-diagram.svg")

Why this architecture:

- **Marketplace** — instant access to curated, live datasets. No ETL, no copies.
- **Iceberg** — open format on your GCS bucket. Any engine reads the same files. No lock-in.
- **Semantic View** — business logic defined once in the data layer. Less hallucination, higher accuracy.
- **Cortex Agent** — natural language → governed SQL → grounded answers.
- **Gemini** — powers the agent with multimodal reasoning, massive context windows, and native GCP integration.
- **MCP** — open standard protocol. Expose once, connect from any AI client.
- **CoWork** — chat interface for business users to get insights, reports, and charts with grounded accuracy.
- **Gemini Enterprise** — scalable corporate AI assistant employees already use daily. Connects to data tools via MCP — no new UI to learn.

## Setup

We need three environments:

- **Snowflake** — our enterprise data warehouse. Hosts Iceberg tables, Semantic Views, Cortex Agents, and MCP servers.
  - Register at [data ops](https://go.dataops.live/snowflake-and-gemini-workshop), then open your Snowflake instance with the provided username and password.
- **GCP** — we create GCS buckets for Iceberg storage and use Gemini Enterprise to register the MCP connection.
  - Open [Qwiklabs](https://explore.qwiklabs.com) for your GCP lab environment. Then, log in or sign up using the same email you used when you registered for the workshop.
- **Looker** — we build BI dashboards on the same Iceberg data.
  - Log in informaiton witll be provided during the workshop.

## Workspace

Snowflake Workspaces give you a full developer environment in the browser — connected to Git, mixing Python and SQL in notebooks, with profiling, lineage, and CoCo assistance built in.

Let's open a new workspace connected to [this repo](https://github.com/sfc-gh-akhosro/gcp-snowflake-solutions), then open `hands-on-lab-cortex-gemini/hol-cortex-gemini.ipynb` and start the service connection. It takes a few minutes — start it now while reading ahead.

> Open a **second browser tab** with the same [Snowflake instance](https://app.snowflake.com). Use that tab to explore Snowflake components (Marketplace, AI & ML, etc.). The first tab stays on the notebook.

Throughout this lab we use two roles — one for the developer (us) and one for the end user who accesses the agent through CoWork or Gemini Enterprise:

- **`hol_role`** — runs this notebook and owns all underlying resources.
- **`end_user_role`** — simulates a business user consuming the agent.

Let's create those roles and grant the needed privileges as our first cell. Take a moment to explore the notebook toolbar — run, stop, and cell controls.

In [ ]:
%%sql -r setup_result
USE ROLE ACCOUNTADMIN;

-- Create a warehouse for this lab (guaranteed to exist)
CREATE WAREHOUSE IF NOT EXISTS hol_wh
  WAREHOUSE_SIZE = 'XSMALL' AUTO_SUSPEND = 60 INITIALLY_SUSPENDED = TRUE;
USE WAREHOUSE hol_wh;

-- Builder role: owns all workshop objects
CREATE ROLE IF NOT EXISTS hol_role;

-- Consumer role: can only use the agent (CoWork, Gemini Enterprise)
CREATE ROLE IF NOT EXISTS end_user_role;

-- Grant both roles to whoever is running this notebook
BEGIN
  LET usr := CURRENT_USER();
  EXECUTE IMMEDIATE 'GRANT ROLE hol_role TO USER ' || :usr;
  EXECUTE IMMEDIATE 'GRANT ROLE end_user_role TO USER ' || :usr;
END;

-- Builder privileges
GRANT CREATE DATABASE        ON ACCOUNT TO ROLE hol_role;
GRANT CREATE WAREHOUSE       ON ACCOUNT TO ROLE hol_role;
GRANT CREATE INTEGRATION     ON ACCOUNT TO ROLE hol_role;
GRANT CREATE EXTERNAL VOLUME ON ACCOUNT TO ROLE hol_role;
GRANT OWNERSHIP ON WAREHOUSE hol_wh TO ROLE hol_role COPY CURRENT GRANTS;

-- Cortex access for both roles
GRANT DATABASE ROLE SNOWFLAKE.CORTEX_USER TO ROLE hol_role;
GRANT DATABASE ROLE SNOWFLAKE.CORTEX_USER TO ROLE end_user_role;

-- Switch to hol_role to build
USE ROLE hol_role;
USE WAREHOUSE hol_wh;

CREATE DATABASE IF NOT EXISTS hol_db;
USE SCHEMA hol_db.public;

-- Schema-level privileges (must come after database/schema exist)
USE ROLE ACCOUNTADMIN;
GRANT CREATE SEMANTIC VIEW ON SCHEMA hol_db.public TO ROLE hol_role;
USE ROLE hol_role;
USE WAREHOUSE hol_wh;
USE SCHEMA hol_db.public;

-- Grant consumer role usage on warehouse and database
GRANT USAGE ON WAREHOUSE hol_wh TO ROLE end_user_role;
GRANT USAGE ON DATABASE hol_db TO ROLE end_user_role;
GRANT USAGE ON SCHEMA hol_db.public TO ROLE end_user_role;

-- Verify context
SELECT CURRENT_ROLE() AS role, CURRENT_WAREHOUSE() AS wh, CURRENT_DATABASE() AS db, CURRENT_SCHEMA() AS schema;

In [ ]:
spotlight("arch-diagram.svg", left=3, top=8, right=26, bottom=92)

## Marketplace

We get our source data from the Snowflake Marketplace. Marketplace gives teams instant access to curated, live datasets — no ETL, no copies, no ingestion pipelines. For data providers, it's a secure, managed channel to distribute data.

We want to build an economic dataset tracking the financial wellbeing of Americans at the state level: income, inflation, mortgage rates, unemployment — monthly. We need four source tables from the Bureau of Labor Statistics and related public data.

Let's go get them.

**UI:** Data Products → Marketplace → search "Snowflake Public Data" → Get (free).

The cell below auto-detects the database name (varies by account) and verifies access.

In [ ]:
%%sql -r marketplace_check
-- Verify marketplace data access
SELECT 'BLS_PRICE' AS source, COUNT(*) AS row_count FROM {{marketplace_db}}.PUBLIC_DATA_FREE.BUREAU_OF_LABOR_STATISTICS_PRICE_TIMESERIES
UNION ALL
SELECT 'BLS_EMPLOYMENT', COUNT(*) FROM {{marketplace_db}}.PUBLIC_DATA_FREE.BUREAU_OF_LABOR_STATISTICS_EMPLOYMENT_TIMESERIES
UNION ALL
SELECT 'FREDDIE_MAC', COUNT(*) FROM {{marketplace_db}}.PUBLIC_DATA_FREE.FREDDIE_MAC_HOUSING_TIMESERIES
UNION ALL
SELECT 'IRS_INCOME', COUNT(*) FROM {{marketplace_db}}.PUBLIC_DATA_FREE.IRS_INDIVIDUAL_INCOME_TIMESERIES;

In [ ]:
spotlight("arch-diagram.svg", left=4, top=15, right=25, bottom=72)

## Iceberg

Now we need somewhere to land this data. We use Apache Iceberg — an open table format where Parquet files and metadata sit in the customer's own GCS bucket. The customer owns the data. Any engine that speaks Iceberg (Snowflake, BigQuery, Spark, Agent Platform) reads the same files directly. No copies between systems, no lock-in. Snowflake manages table metadata through the Horizon Catalog.

Let's create the bucket, give Snowflake write access, and build our economic indicators table.

**UI:** Google Cloud Console → Cloud Storage → Create Bucket → name: `firstname_lastname_hol_0729` → region: `Multi-region`.

In [ ]:
%%sql -r ext_vol_result
-- Create an external volume pointing to your GCS bucket
CREATE EXTERNAL VOLUME IF NOT EXISTS hol_gcs_vol
  STORAGE_LOCATIONS = ((
    NAME = 'hol-gcs'
    STORAGE_PROVIDER = 'GCS'
    STORAGE_BASE_URL = 'gcs://{{bucket_name}}/iceberg/'
  ));

-- Describe to get storage config
DESCRIBE EXTERNAL VOLUME hol_gcs_vol;
SET desc_qid = LAST_QUERY_ID();

-- Extract the GCS service account to grant on the bucket
SELECT
  PARSE_JSON("property_value"):STORAGE_GCP_SERVICE_ACCOUNT::STRING
    AS gcs_service_account_to_grant
FROM TABLE(RESULT_SCAN($desc_qid))
WHERE "property" = 'STORAGE_LOCATION_1';

In [ ]:
%%sql -r iceberg_table
-- Create Iceberg table: join four marketplace sources into one wide-format table
CREATE OR REPLACE ICEBERG TABLE hol_db.public.economic_indicators
  CATALOG = 'SNOWFLAKE'
  EXTERNAL_VOLUME = 'hol_gcs_vol'
  BASE_LOCATION = 'economic_indicators'
  AS
WITH cpi AS (
  SELECT
    DATE_TRUNC('month', date) AS month,
    AVG(value) AS cpi_index
  FROM {{marketplace_db}}.PUBLIC_DATA_FREE.BUREAU_OF_LABOR_STATISTICS_PRICE_TIMESERIES
  WHERE variable_name = 'CPI: All items, Not seasonally adjusted, Monthly'
    AND geo_id = 'country/USA'
  GROUP BY 1
),
mortgage_30yr AS (
  SELECT
    DATE_TRUNC('month', date) AS month,
    ROUND(AVG(value) * 100, 2) AS mortgage_rate_30yr_pct
  FROM {{marketplace_db}}.PUBLIC_DATA_FREE.FREDDIE_MAC_HOUSING_TIMESERIES
  WHERE variable_name = '30-Year Fixed Rate Mortgage Rate, National Average'
    AND geo_id = 'country/USA'
  GROUP BY 1
),
mortgage_15yr AS (
  SELECT
    DATE_TRUNC('month', date) AS month,
    ROUND(AVG(value) * 100, 2) AS mortgage_rate_15yr_pct
  FROM {{marketplace_db}}.PUBLIC_DATA_FREE.FREDDIE_MAC_HOUSING_TIMESERIES
  WHERE variable_name = '15-Year Fixed Rate Mortgage Rate, National Average'
    AND geo_id = 'country/USA'
  GROUP BY 1
),
unemployment AS (
  SELECT
    DATE_TRUNC('month', date) AS month,
    geo_id,
    AVG(value) AS unemployment_rate_pct
  FROM {{marketplace_db}}.PUBLIC_DATA_FREE.BUREAU_OF_LABOR_STATISTICS_EMPLOYMENT_TIMESERIES
  WHERE variable_name = 'Local Area Unemployment: Unemployment Rate, Not seasonally adjusted, Monthly'
    AND LENGTH(geo_id) = 8
  GROUP BY 1, 2
),
national_unemployment AS (
  SELECT month, ROUND(AVG(unemployment_rate_pct), 2) AS unemployment_rate_pct
  FROM unemployment
  GROUP BY 1
),
income_raw AS (
  SELECT
    agi.geo_id,
    YEAR(agi.date) AS yr,
    ROUND(agi.value / NULLIF(ret.value, 0), 0) AS avg_income_per_return
  FROM {{marketplace_db}}.PUBLIC_DATA_FREE.IRS_INDIVIDUAL_INCOME_TIMESERIES agi
  JOIN {{marketplace_db}}.PUBLIC_DATA_FREE.IRS_INDIVIDUAL_INCOME_TIMESERIES ret
    ON agi.geo_id = ret.geo_id AND agi.date = ret.date
  WHERE agi.variable_name = 'Adjusted gross income (AGI), AGI bin: Total'
    AND ret.variable_name = 'Number of returns, AGI bin: Total'
    AND LENGTH(agi.geo_id) = 8
),
income_indexed AS (
  SELECT
    geo_id,
    yr,
    avg_income_per_return,
    ROUND((avg_income_per_return / FIRST_VALUE(avg_income_per_return) OVER (PARTITION BY geo_id ORDER BY yr)) * 100, 1) AS income_index
  FROM income_raw
),
national_income AS (
  SELECT yr, ROUND(AVG(income_index), 1) AS income_index
  FROM income_indexed
  GROUP BY 1
),
geo AS (
  SELECT geo_id, geo_name
  FROM {{marketplace_db}}.PUBLIC_DATA_FREE.GEOGRAPHY_INDEX
  WHERE level = 'State'
),
national AS (
  SELECT
    c.month AS date,
    'country/USA' AS geo_id,
    'United States' AS geo_name,
    ROUND(c.cpi_index, 2) AS cpi_index,
    ROUND(((c.cpi_index - LAG(c.cpi_index, 12) OVER (ORDER BY c.month))
      / NULLIF(LAG(c.cpi_index, 12) OVER (ORDER BY c.month), 0)) * 100, 2) AS inflation_pct,
    m30.mortgage_rate_30yr_pct,
    m15.mortgage_rate_15yr_pct,
    nu.unemployment_rate_pct,
    ni.income_index
  FROM cpi c
  LEFT JOIN mortgage_30yr m30 ON c.month = m30.month
  LEFT JOIN mortgage_15yr m15 ON c.month = m15.month
  LEFT JOIN national_unemployment nu ON c.month = nu.month
  LEFT JOIN national_income ni ON YEAR(c.month) = ni.yr
),
states AS (
  SELECT
    u.month AS date,
    u.geo_id,
    g.geo_name,
    NULL::FLOAT AS cpi_index,
    NULL::FLOAT AS inflation_pct,
    NULL::FLOAT AS mortgage_rate_30yr_pct,
    NULL::FLOAT AS mortgage_rate_15yr_pct,
    u.unemployment_rate_pct,
    ii.income_index
  FROM unemployment u
  JOIN geo g ON u.geo_id = g.geo_id
  LEFT JOIN income_indexed ii ON u.geo_id = ii.geo_id AND YEAR(u.month) = ii.yr
)
SELECT * FROM national
UNION ALL
SELECT * FROM states
ORDER BY date, geo_id;

In [ ]:
spotlight("arch-diagram.svg", left=26, top=12, right=45, bottom=68)

## Exploration

Let's explore our data. Snowsight provides query profiling, lineage, charting, and pivot tables right in each cell — helping you understand the dataset at a glance.

Run the query below, then try the **Chart** tab and **Query Profile** to see how data flows.

In [ ]:
%%sql -r profiling_data
-- National economic indicators since 2015
-- Try: Chart (line) to visualize trends, Query Profile to see execution plan
SELECT
  date,
  cpi_index,
  income_index,
  inflation_pct,
  mortgage_rate_30yr_pct,
  unemployment_rate_pct
FROM hol_db.public.economic_indicators
WHERE geo_id = 'country/USA'
  AND date >= '2015-01-01'
  AND inflation_pct IS NOT NULL
ORDER BY date;

## Cortex

We have a clean Iceberg table. Any analyst can query it with SQL. But that's not AI-ready yet.

The gap between "data is queryable" and "AI gives accurate answers" is semantic context. An LLM looking at column names like `CPI_INDEX` or `GEO_ID` will guess — and hallucinate. We need to tell it what the data means: which columns are dimensions, which are facts, how metrics are calculated, what questions this table answers.

That's what a Semantic View does. It defines business logic once — in the data layer, not scattered across prompts. Every AI consumer inherits the same correct definitions.

In [ ]:
spotlight("arch-diagram.svg", left=45, top=30, right=75, bottom=80)

### Semantic View

The Semantic View is the grounding layer for Cortex Analyst. We define dimensions (date, geography), facts (CPI, mortgage rate, unemployment, income), and metrics (year-over-year inflation, average mortgage rate by state). We can also add verified queries — known-good question-to-SQL mappings that anchor the model's behavior.

Without this, an LLM guesses. With this, "How has inflation compared to income growth?" maps to the exact right expressions every time.

In [ ]:
%%sql -r semantic_view_result
CALL SYSTEM$CREATE_SEMANTIC_VIEW_FROM_YAML(
  'hol_db.public',
  $$
name: economic_semantic_view
tables:
  - name: economic_indicators
    base_table:
      database: HOL_DB
      schema: PUBLIC
      table: ECONOMIC_INDICATORS
    dimensions:
      - name: DATE
        description: "Date of the observation"
        expr: economic_indicators.DATE
        data_type: DATE
      - name: GEO_ID
        description: "Geographic area identifier"
        expr: economic_indicators.GEO_ID
        data_type: TEXT
      - name: GEO_NAME
        description: "Geographic area — United States for national, or state name (e.g. California)"
        expr: economic_indicators.GEO_NAME
        data_type: TEXT
    facts:
      - name: CPI_INDEX
        description: "Consumer Price Index, base period 1982-84 = 100 (national only)"
        expr: economic_indicators.CPI_INDEX
        data_type: NUMBER
      - name: INFLATION_PCT
        description: "Year-over-year inflation rate as percent (national only)"
        expr: economic_indicators.INFLATION_PCT
        data_type: NUMBER
      - name: MORTGAGE_RATE_30YR_PCT
        description: "30-year fixed mortgage rate, national average, percent (national only)"
        expr: economic_indicators.MORTGAGE_RATE_30YR_PCT
        data_type: NUMBER
      - name: MORTGAGE_RATE_15YR_PCT
        description: "15-year fixed mortgage rate, national average, percent (national only)"
        expr: economic_indicators.MORTGAGE_RATE_15YR_PCT
        data_type: NUMBER
      - name: UNEMPLOYMENT_RATE_PCT
        description: "Unemployment rate as percent (available national and by state)"
        expr: economic_indicators.UNEMPLOYMENT_RATE_PCT
        data_type: NUMBER
      - name: INCOME_INDEX
        description: "Average income per tax return, indexed to earliest available year = 100. Compare to CPI_INDEX to assess purchasing power. (available national and by state, annual grain)"
        expr: economic_indicators.INCOME_INDEX
        data_type: NUMBER
    metrics:
      - name: AVG_CPI_INDEX
        description: "Average Consumer Price Index"
        expr: AVG(economic_indicators.CPI_INDEX)
      - name: AVG_INFLATION_PCT
        description: "Average year-over-year inflation rate"
        expr: AVG(economic_indicators.INFLATION_PCT)
      - name: AVG_MORTGAGE_RATE_30YR
        description: "Average 30-year fixed mortgage rate"
        expr: AVG(economic_indicators.MORTGAGE_RATE_30YR_PCT)
      - name: AVG_UNEMPLOYMENT_RATE
        description: "Average unemployment rate"
        expr: AVG(economic_indicators.UNEMPLOYMENT_RATE_PCT)
      - name: AVG_INCOME_INDEX
        description: "Average income index"
        expr: AVG(economic_indicators.INCOME_INDEX)
$$
);

-- Verify
SHOW SEMANTIC VIEWS IN SCHEMA hol_db.public;

### Cortex Agent

Now we wrap the Semantic View in a conversational interface. A Cortex Agent takes a natural-language question, routes it through Cortex Analyst (which uses the Semantic View to generate correct SQL), executes it, and returns a grounded answer with supporting data.

The agent is powered by Gemini as the reasoning model. We define it in SQL — reproducible and version-controlled.

**UI:** You can also create agents visually at AI & ML → Cortex Agents.

In [ ]:
%%sql -r agent_result
-- Create a Cortex Agent backed by the economic indicators semantic view
CREATE OR REPLACE AGENT hol_db.public.hol_economic_agent
  FROM SPECIFICATION $$
  tools:
    - tool_spec:
        type: cortex_analyst_text_to_sql
        name: economic_analyst
        description: "Answers questions about US economic indicators: CPI/inflation, mortgage interest rates (30-year, 15-year), unemployment rate (national and by state), and income index."

  tool_resources:
    economic_analyst:
      semantic_view: HOL_DB.PUBLIC.ECONOMIC_SEMANTIC_VIEW
      execution_environment:
        type: warehouse
        warehouse: HOL_WH
  $$;

-- Grant consumer role usage on the agent
GRANT USAGE ON AGENT hol_db.public.hol_economic_agent TO ROLE end_user_role;
GRANT SELECT ON SEMANTIC VIEW hol_db.public.economic_semantic_view TO ROLE end_user_role;
GRANT SELECT ON TABLE hol_db.public.economic_indicators TO ROLE end_user_role;

-- Verify
SHOW AGENTS IN SCHEMA hol_db.public;

In [ ]:
spotlight("arch-diagram.svg", left=75, top=30, right=98, bottom=80)

### CoWork

CoWork is the chat surface for business users. No SQL, no notebook — just a conversation with the agent. We switch to `end_user_role` to simulate a business user who can only consume, not build. Ask the same economic question and inspect the generated SQL in the response.

Same agent, same data, different role, chat-based surface.

**UI:** AI & ML → Snowflake Intelligence → switch role to `end_user_role` / `hol_wh` → select **hol_economic_agent** → ask:

*"How has the 30-year mortgage rate changed relative to inflation since 2020?"*

## MCP

So far the agent lives inside Snowflake. To make it accessible to external AI clients, we expose it via MCP — Model Context Protocol. MCP is an open standard (started by Anthropic, now under the Linux Foundation) that gives AI applications a universal interface to data tools. Declare the agent as an MCP tool, add OAuth for secure access, done. Any MCP-compatible client connects through the same protocol — no custom integrations per client.

We create a Snowflake-managed MCP server with the agent registered as a callable tool, then set up OAuth so external clients can authenticate securely. The output gives us the credentials we register in Gemini Enterprise next.

In [ ]:
%%sql -r mcp_credentials
-- MCP server exposing the Cortex Agent as a tool
CREATE OR REPLACE MCP SERVER hol_db.public.hol_mcp
  FROM SPECIFICATION $$
  tools:
    - name: "hol-economic-agent"
      type: "CORTEX_AGENT_RUN"
      identifier: "HOL_DB.PUBLIC.HOL_ECONOMIC_AGENT"
      description: "US economic indicators agent — answers questions about inflation (CPI), mortgage rates, unemployment, and income."
      title: "Economic Indicators Agent"
  $$;

-- OAuth security integration for external MCP clients
USE ROLE ACCOUNTADMIN;
CREATE OR REPLACE SECURITY INTEGRATION hol_mcp_oauth
  TYPE = OAUTH
  OAUTH_CLIENT = CUSTOM
  OAUTH_CLIENT_TYPE = 'CONFIDENTIAL'
  OAUTH_REDIRECT_URI = 'https://vertexaisearch.cloud.google.com/oauth-redirect'
  ENABLED = TRUE;
USE ROLE hol_role;
USE WAREHOUSE hol_wh;
USE SCHEMA hol_db.public;

-- Retrieve all credentials needed for Gemini Enterprise MCP connection
WITH secrets AS (
  SELECT PARSE_JSON(SYSTEM$SHOW_OAUTH_CLIENT_SECRETS('HOL_MCP_OAUTH')) AS s
),
account_url AS (
  SELECT 'https://' || CURRENT_ORGANIZATION_NAME() || '-' || CURRENT_ACCOUNT_NAME() || '.snowflakecomputing.com' AS base
)
SELECT field_name, value
FROM (
  SELECT 1 AS ord, 'MCP Server URL'   AS field_name, a.base || '/api/v2/cortex/mcp' AS value FROM account_url a
  UNION ALL
  SELECT 2, 'Auth URL',              a.base || '/oauth/authorize' FROM account_url a
  UNION ALL
  SELECT 3, 'Auth URL Params',       '' FROM account_url a
  UNION ALL
  SELECT 4, 'Token URL',             a.base || '/oauth/token-request' FROM account_url a
  UNION ALL
  SELECT 5, 'Client ID',             s.s:OAUTH_CLIENT_ID::STRING FROM secrets s
  UNION ALL
  SELECT 6, 'Client Secret',         s.s:OAUTH_CLIENT_SECRET::STRING FROM secrets s
  UNION ALL
  SELECT 7, 'Scopes',                'session:role:end_user_role' FROM secrets s
  UNION ALL
  SELECT 8, 'MCP Server Description', 'Snowflake Cortex Agent for US economic indicators (CPI, mortgage rates, unemployment, income)' FROM secrets s
  UNION ALL
  SELECT 9, 'Agent Instructions',    'Use the hol-economic-agent tool to answer questions about US economic data including inflation, mortgage rates, unemployment by state, and income trends.' FROM secrets s
  UNION ALL
  SELECT 10, 'Data Connector Name',  'hol_cortex_gemini_economic_agent' FROM secrets s
)
ORDER BY ord;

## Gemini Enterprise

Gemini Enterprise is Google Cloud's corporate AI assistant — the chat interface employees already use daily. By registering our Snowflake MCP server as a data connector, the Cortex Agent becomes a tool Gemini calls natively. Employees ask questions in Gemini and get grounded answers from governed Iceberg data — without knowing anything about Snowflake or SQL underneath.

Same question, same answer, different surface. Build the agent once, consume it everywhere.

**UI:** Google Cloud Console → Gemini for Google Cloud → Data Connectors → Add Connector → Custom MCP Server → fill values from OAuth output above → complete OAuth flow → Enable Actions.

Then in Gemini Enterprise chat, ask:

*"How has the 30-year mortgage rate changed relative to inflation since 2020?"*

### Troubleshooting

**Network Policy** — If Gemini can't reach Snowflake (OAuth errors, timeouts), a network policy may be blocking external IPs. Run the cell below to temporarily allow all connections.

**Google Cloud Org Policy** — If you see `constraints/discoveryengine.managed.disableCustomMcpServerConnector`:

**UI:** IAM & Admin → Organization Policies → search `disableCustomMcpServerConnector` → Enforcement: Off → Save → retry.

In [ ]:
%%sql -r troubleshoot_result
-- Temporarily disable account network policy to allow Gemini Enterprise OAuth
USE ROLE ACCOUNTADMIN;
ALTER ACCOUNT UNSET NETWORK_POLICY;

-- To re-enable later:
-- ALTER ACCOUNT SET NETWORK_POLICY = <your_policy_name>;

## Looker

The same Iceberg data that powers the agent also feeds traditional BI. Looker connects directly to the Snowflake table — no additional copies or pipelines. One data product: governed dashboards alongside AI chat.

**UI (Looker):** Admin → Database → Connections → add Snowflake connection (use lab credentials, `HOL_DB.PUBLIC`) → create LookML project on `ECONOMIC_INDICATORS` → build Explore + Dashboard.

> Detailed steps in Qwiklabs instructions.

## Wrap-up

One copy of data on open Iceberg in your GCS bucket. A Semantic View that grounds AI in business logic. A Cortex Agent that turns questions into governed SQL. Consumed from CoWork, Gemini Enterprise, Looker, and any MCP client.

No copies. No custom integrations per surface. No hallucination from ungrounded prompts. Build once, consume everywhere.

In [ ]:
spotlight("arch-diagram.svg")

## Cleanup

Run only when you're done with the lab.

In [ ]:
%%sql -r cleanup_result
USE ROLE ACCOUNTADMIN;

-- Drop database (cascades all objects inside: tables, views, agents, MCP servers)
DROP DATABASE IF EXISTS hol_db;
DROP WAREHOUSE IF EXISTS hol_wh;
DROP INTEGRATION IF EXISTS hol_mcp_oauth;
DROP ROLE IF EXISTS hol_role;
DROP ROLE IF EXISTS end_user_role;

-- NOTE: hol_gcs_vol is kept — GCS bucket permissions take time to set up
-- To drop it manually: DROP EXTERNAL VOLUME IF EXISTS hol_gcs_vol;

-- Re-enable network policy if it was disabled
-- ALTER ACCOUNT SET NETWORK_POLICY = ACCOUNT_VPN_POLICY_SE;

SHOW ROLES LIKE '%HOL%';

In [ ]:
# Detect the Snowflake Public Data database name (varies by account)
import snowflake.snowpark.context
session = snowflake.snowpark.context.get_active_session()

candidates = ['SNOWFLAKE_PUBLIC_DATA', 'SNOWFLAKE_PUBLIC_DATA_FREE']
marketplace_db = None
for name in candidates:
    try:
        session.sql(f"SELECT 1 FROM {name}.PUBLIC_DATA_FREE.BUREAU_OF_LABOR_STATISTICS_PRICE_TIMESERIES LIMIT 1").collect()
        marketplace_db = name
        break
    except:
        continue

if marketplace_db is None:
    raise RuntimeError("Marketplace database not found. Go to Data Products → Marketplace → search 'Snowflake Public Data' → Get (free).")

print(f"✓ Using marketplace database: {marketplace_db}")

# Set as session variable for SQL cells
session.sql(f"SET marketplace_db = '{marketplace_db}'").collect()

In [ ]:
# ✏️ Set your GCS bucket name (the one you created in Google Cloud Console)
bucket_name = "hands-on-lab-cortex-gemini"  # <-- CHANGE THIS